# Lesson 5: Optimizers, Regularization & PyTorch Training Loops

**Runtime Tier**: Google Colab Free Tier **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`)

Compare AdamW in raw NumPy with a GPU-ready PyTorch `nn.Module` training loop.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active PyTorch device:', device)


In [ ]:
# Synthetic non-linear spiral/circle classification dataset
rng = np.random.default_rng(42)
n = 1000
X_np = rng.normal(size=(n, 2)).astype(np.float32)
y_np = ((X_np[:, 0] ** 2 + X_np[:, 1] ** 2) > 1.2).astype(np.int64)

ds = TensorDataset(torch.from_numpy(X_np), torch.from_numpy(y_np))
loader = DataLoader(ds, batch_size=64, shuffle=True)

model = nn.Sequential(
    nn.Linear(2, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(64, 2),
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(15):
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | Loss: {total_loss / n:.4f}')

model.eval()
with torch.no_grad():
    preds = model(torch.from_numpy(X_np).to(device)).argmax(dim=1).cpu().numpy()
print('Train Accuracy:', round(float((preds == y_np).mean()), 4))
